# 01 — Export random-window data (u, v, ssh, bathymetry)

One export supports **all training modes** via `channel_indices` in train notebooks:

| Mode | `TRAINING_MODE` | export indices | predict |
|---|---|---|---|
| uv → uv | `"uv"` | `[0, 1]` | u, v |
| uvssh → uvssh | `"uvssh"` | `[0, 1, 2]` | u, v, ssh |
| uvbathy → uvbathy | `"uvbathy"` | `[0, 1, 3]` | u, v, bathymetry |
| uvsshbathy → uvsshbathy | `"uvsshbathy"` | `[0, 1, 2, 3]` | u, v, ssh, bathymetry |

**Export channel order (fixed):** `(u, v, ssh, bathymetry)` → indices `0, 1, 2, 3`

**Settings**
- `DT = 360`, 3 cond + 1 pred
- **Pool:** noon-only (`HOUR_OF_DAY = 12`), **K=2 crops per anchor**
- **Window pool:** 12 cached crops/row → `{prefix}_sims_pool_k1.npy` (no Zarr during training)
- **Mask filter:** land/invalid **≥ 5%** rejected
- **Test:** 2023 noon full year, 1 crop per anchor
- **Device:** `cuda:3` for Zarr preview; export subprocess uses `CUDA_VISIBLE_DEVICES=3`

**Output** — `Shalow/data/{export_key}/`:
```
sample_13_22_uvsshbathy_dt360_*
sample_13_22_uvsshbathy_dt360_sims_pool_k1.npy
sample_13_22_export.json
sample_23_uvsshbathy_dt360_*
```


In [1]:
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
SHALOW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Shalow" else NOTEBOOK_DIR / "Shalow"
ROOT = SHALOW_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SHALOW_DIR) not in sys.path:
    sys.path.insert(0, str(SHALOW_DIR))

DATA_DIR = SHALOW_DIR / "data"
LOG_DIR = SHALOW_DIR / "logs"
for _d in (DATA_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.export.fix_window_export import candidate_indices_for_hour
from src.export.random_window_export import (
    RandomWindowExportConfig,
    export_wants_fresh_start,
    plan_multi_crop_export,
    run_export,
)
from src.train.channel_select import EXPORT_UVSSH_BATHY_NAMES, TRAINING_MODES
from window_paths import export_manifest_path, pool_prefix, test_prefix, window_data_dir


In [2]:
ZARR_PATH = os.getenv(
    "GUIANA_ZARR_PATH",
    "s3://oceangen-data/margem_equatorial_tensor.zarr",
)

# Dedicated export key — separate from uv-only random_win_128 exports
EXPORT_KEY = "random_win_128_uvsshbathy"
EXPORT_DEVICE = "cuda:3"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

CROP_HEIGHT = 128
CROP_WIDTH = 128

DT = 360
CONDITIONING_STEPS = 3
FORECAST_STEPS = 1
EXPORT_COMPONENTS = ("uvsshbathy",)

WINDOW_DATA_DIR = window_data_dir(DATA_DIR, EXPORT_KEY)
POOL_DATA_PREFIX_BASE = pool_prefix(DATA_DIR, EXPORT_KEY, component="uvsshbathy", dt=DT)
TEST_DATA_PREFIX_BASE = test_prefix(DATA_DIR, EXPORT_KEY, component="uvsshbathy", dt=DT)
EXPORT_MANIFEST_PATH = export_manifest_path(DATA_DIR, EXPORT_KEY)


def _component_export_settings(component: str, base: str | None = None):
    if component == "uvsshbathy":
        base = base or POOL_DATA_PREFIX_BASE
        # Full 4-channel export; training picks subsets via channel_indices / TRAINING_MODES
        return base, EXPORT_UVSSH_BATHY_NAMES, (0, 1, 2, 3)
    raise ValueError(f"Unknown component {component!r}; use 'uvsshbathy'")


PREVIEW_COMPONENT = EXPORT_COMPONENTS[0]
OUTPUT_PREFIX, CHANNEL_NAMES, TARGET_CHANNEL_INDICES = _component_export_settings(PREVIEW_COMPONENT)

SEED = 42
N_JOBS = 30
MAX_MASK_FRAC = 0.05
MAX_CROP_ATTEMPTS = 200
HOUR_OF_DAY = 12
CROPS_PER_ANCHOR = 2

EXPORT_WINDOW_POOL = True
N_WINDOWS_PER_SAMPLE = 12
TRAIN_CROPS_FOR_WINDOW_POOL = (1, 2)
EXPORT_EPOCH_WINDOWS = False

N_EXPORT_EPOCHS = 100
RANDOM_CROP_SEED = 42
VAL_SPLIT = 0.3
VAL_SPLIT_MODE = "random"
VAL_SPLIT_SEED = 42
TRAIN_CROPS_FOR_EPOCH_WINDOWS = (1, 2)

N_SAMPLES_TO_GEN = 8000

START_TIME = "2013-01-01"
END_TIME = "2022-12-31"

WINDOW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Export device:", EXPORT_DEVICE)
print("Export channels:", CHANNEL_NAMES, "→ indices", list(range(len(CHANNEL_NAMES))))
print("Training modes from this export:")
for mode, preset in TRAINING_MODES.items():
    print(f"  {mode:12s} channel_indices={list(preset['channel_indices'])}")


Export device: cuda:3
Export channels: ('u', 'v', 'ssh', 'bathymetry') → indices [0, 1, 2, 3]
Training modes from this export:
  uv           channel_indices=[0, 1]
  uvssh        channel_indices=[0, 1, 2]
  uvbathy      channel_indices=[0, 1, 3]
  uvsshbathy   channel_indices=[0, 1, 2, 3]


In [3]:
base_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
    start_time=START_TIME,
    end_time=END_TIME,
)

print("Opening Zarr:", ZARR_PATH, "| device:", EXPORT_DEVICE)
dataset = GuianaDataloader(**base_kwargs, transform=None, device=EXPORT_DEVICE)
info = dataset.info()
candidate_idx = candidate_indices_for_hour(dataset, HOUR_OF_DAY)

pool_anchors, pool_zarr_idx, pool_crop_slots = plan_multi_crop_export(
    candidate_idx,
    n_samples_to_gen=N_SAMPLES_TO_GEN,
    crops_per_anchor=CROPS_PER_ANCHOR,
    seed=SEED,
)
n_anchors = len(pool_anchors)
n_rows = len(pool_zarr_idx)

export_manifest = {
    "export_key": EXPORT_KEY,
    "export_components": list(EXPORT_COMPONENTS),
    "channel_names": list(CHANNEL_NAMES),
    "export_channel_order": list(CHANNEL_NAMES),
    "training_modes": {
        mode: {
            "channel_indices": list(preset["channel_indices"]),
            "channel_names": list(preset["channel_names"]),
        }
        for mode, preset in TRAINING_MODES.items()
    },
    "window_data_dir": str(WINDOW_DATA_DIR),
    "window_mode": "random",
    "mode": "random_pool_export",
    "sample_seed": SEED,
    "n_pool": int(len(candidate_idx)),
    "n_anchors": int(n_anchors),
    "crops_per_anchor": int(CROPS_PER_ANCHOR),
    "n_rows": int(n_rows),
    "start_time": START_TIME,
    "end_time": END_TIME,
    "hour_of_day": HOUR_OF_DAY,
    "crop_height": CROP_HEIGHT,
    "crop_width": CROP_WIDTH,
    "max_mask_frac": MAX_MASK_FRAC,
    "pool_data_prefix_base": POOL_DATA_PREFIX_BASE,
    "train_val_split_note": "70/30 split at training time; set TRAINING_MODE + channel_indices in 04 train notebook",
    "n_export_epochs": N_EXPORT_EPOCHS,
    "random_crop_seed": RANDOM_CROP_SEED,
    "val_split": VAL_SPLIT,
    "val_split_mode": VAL_SPLIT_MODE,
    "val_split_seed": VAL_SPLIT_SEED,
    "export_window_pool": EXPORT_WINDOW_POOL,
    "n_windows_per_sample": N_WINDOWS_PER_SAMPLE,
    "train_crops_for_window_pool": list(TRAIN_CROPS_FOR_WINDOW_POOL),
    "export_epoch_windows": EXPORT_EPOCH_WINDOWS,
    "train_crops_for_epoch_windows": list(TRAIN_CROPS_FOR_EPOCH_WINDOWS),
    "zarr_indices": pool_anchors.tolist(),
}
EXPORT_MANIFEST_PATH.write_text(json.dumps(export_manifest, indent=2))

print("Valid samples:", len(dataset))
print("After hour filter:", len(candidate_idx), "| HOUR_OF_DAY =", HOUR_OF_DAY)
print(
    f"Pool plan: {n_anchors} noon anchors × {CROPS_PER_ANCHOR} crops = {n_rows} export rows "
    f"(seed={SEED}) → {POOL_DATA_PREFIX_BASE}"
)
print("Export manifest:", EXPORT_MANIFEST_PATH)
print("Grid (H, W):", info["space"]["height"], info["space"]["width"])
print(f"Crop: {CROP_HEIGHT}x{CROP_WIDTH} random | keep if land/invalid < {MAX_MASK_FRAC:.0%}")
if EXPORT_WINDOW_POOL:
    print(
        f"Window pool export: {N_WINDOWS_PER_SAMPLE} crops/row, k={TRAIN_CROPS_FOR_WINDOW_POOL} "
        f"(cached sims — no Zarr during training)"
    )
if n_rows < N_SAMPLES_TO_GEN:
    print(
        f"NOTE: {n_rows} rows < N_SAMPLES_TO_GEN={N_SAMPLES_TO_GEN} "
        f"(limited by {len(candidate_idx)} noon anchors × {CROPS_PER_ANCHOR})"
    )


Opening Zarr: s3://oceangen-data/margem_equatorial_tensor.zarr | device: cuda:3


Valid samples: 87648
After hour filter: 3652 | HOUR_OF_DAY = 12
Pool plan: 3652 noon anchors × 2 crops = 7304 export rows (seed=42) → /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_13_22_uvsshbathy_dt360
Export manifest: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_13_22_export.json
Grid (H, W): 431 547
Crop: 128x128 random | keep if land/invalid < 5%
Window pool export: 12 crops/row, k=(1, 2) (cached sims — no Zarr during training)
NOTE: 7304 rows < N_SAMPLES_TO_GEN=8000 (limited by 3652 noon anchors × 2)


## Parallel export (resumable)

One export job for the 2013–2022 pool. Re-run or use the CLI to resume.

After the pool merge finishes, **window pool** export runs automatically (`_sims_pool_k1.npy`, `_k2.npy`). If the pool is already exported:

```bash
python scripts/export_random_window_data.py --output-prefix <POOL_PREFIX> --window-pool-only
```

Legacy per-epoch windows (slow):

```bash
python scripts/export_random_window_data.py --output-prefix <POOL_PREFIX> --epoch-windows-only
```


In [4]:
import subprocess

DETACHED = True
CHUNK_SIZE = 100
RESUME = True


def _run_one_export(
    output_prefix: str,
    channel_names: tuple[str, ...],
    target_channel_indices: tuple[int, ...],
    *,
    start_time: str | None = None,
    end_time: str | None = None,
    n_samples_to_gen: int | None = None,
    crops_per_anchor: int | None = None,
    fixed_dataset_indices: tuple[int, ...] | None = None,
):
    export_config = RandomWindowExportConfig(
        zarr_path=ZARR_PATH,
        dt=DT,
        conditioning_steps=CONDITIONING_STEPS,
        forecast_steps=FORECAST_STEPS,
        crop_height=CROP_HEIGHT,
        crop_width=CROP_WIDTH,
        seed=SEED,
        n_jobs=N_JOBS,
        max_mask_frac=MAX_MASK_FRAC,
        max_crop_attempts=MAX_CROP_ATTEMPTS,
        hour_of_day=HOUR_OF_DAY,
        start_time=start_time or START_TIME,
        end_time=end_time or END_TIME,
        output_prefix=output_prefix,
        n_samples_to_gen=n_samples_to_gen if n_samples_to_gen is not None else N_SAMPLES_TO_GEN,
        crops_per_anchor=crops_per_anchor if crops_per_anchor is not None else CROPS_PER_ANCHOR,
        chunk_size=CHUNK_SIZE,
        channel_names=channel_names,
        target_channel_indices=target_channel_indices,
        fixed_dataset_indices=fixed_dataset_indices,
        export_key=EXPORT_KEY,
        export_epoch_windows=EXPORT_EPOCH_WINDOWS,
        export_window_pool=EXPORT_WINDOW_POOL,
        n_windows_per_sample=N_WINDOWS_PER_SAMPLE,
        train_crops_for_window_pool=TRAIN_CROPS_FOR_WINDOW_POOL,
        n_export_epochs=N_EXPORT_EPOCHS,
        random_crop_seed=RANDOM_CROP_SEED,
        val_split=VAL_SPLIT,
        val_split_mode=VAL_SPLIT_MODE,
        val_split_seed=VAL_SPLIT_SEED,
        train_crops_for_epoch_windows=TRAIN_CROPS_FOR_EPOCH_WINDOWS,
    )

    config_path = Path(f"{output_prefix}_export_config.json")
    config_dict = asdict(export_config)
    config_dict["window_mode"] = "random"
    config_path.write_text(json.dumps(config_dict, indent=2, default=str))
    print(f"Wrote config: {config_path}")

    fresh_start = export_wants_fresh_start(export_config, resume=RESUME)

    if DETACHED:
        log_path = LOG_DIR / f"export_{EXPORT_KEY}_{Path(output_prefix).name}.log"
        cmd = [
            sys.executable,
            str(ROOT / "scripts/export_random_window_data.py"),
            "--output-prefix",
            output_prefix,
        ]
        if fresh_start:
            cmd.append("--no-resume")
            if log_path.exists():
                log_path.unlink()
                print(f"Cleared log: {log_path}")
        log_mode = "w" if fresh_start else "a"
        export_env = os.environ.copy()
        export_env["CUDA_VISIBLE_DEVICES"] = "3"
        with open(log_path, log_mode) as log_file:
            proc = subprocess.Popen(
                cmd,
                stdout=log_file,
                stderr=subprocess.STDOUT,
                start_new_session=True,
                cwd=ROOT,
                env=export_env,
            )
        print(f"Started background export PID {proc.pid} -> {output_prefix}")
        if fresh_start:
            print("Fresh export (log cleared, partial state reset if config changed)")
        else:
            print(f"Resuming export (appending to {log_path})")
        print(f"Log: {log_path}")
        print(f"Monitor: tail -f {log_path}")
        print(
            f"Resume: python scripts/export_random_window_data.py --output-prefix {output_prefix}"
        )
    else:
        run_export(export_config, resume=not fresh_start)


for component in EXPORT_COMPONENTS:
    pool_pfx, ch_names, tgt_idx = _component_export_settings(component, POOL_DATA_PREFIX_BASE)
    print(
        f"\n=== Pool export: {component} -> {pool_pfx} "
        f"({n_anchors} anchors × {CROPS_PER_ANCHOR} = {n_rows} rows) ==="
    )
    _run_one_export(
        pool_pfx,
        ch_names,
        tgt_idx,
        n_samples_to_gen=n_rows,
        crops_per_anchor=CROPS_PER_ANCHOR,
        fixed_dataset_indices=tuple(int(x) for x in pool_anchors),
    )



=== Pool export: uvsshbathy -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_13_22_uvsshbathy_dt360 (3652 anchors × 2 = 7304 rows) ===
Wrote config: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_13_22_uvsshbathy_dt360_export_config.json
Started background export PID 1972853 -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_13_22_uvsshbathy_dt360
Fresh export (log cleared, partial state reset if config changed)
Log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_uvsshbathy_sample_13_22_uvsshbathy_dt360.log
Monitor: tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_uvsshbathy_sample_13_22_uvsshbathy_dt360.log
Resume: python scripts/export_random_window_data.py --output-prefix /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_13_22_uvsshbat

## 2023 test export (rollout holdout)


In [5]:
TEST_START_TIME = "2023-01-01"
TEST_END_TIME = "2023-12-31"
TEST_EXPORT_ALL = True

test_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
    start_time=TEST_START_TIME,
    end_time=TEST_END_TIME,
)
test_dataset = GuianaDataloader(**test_kwargs, transform=None, device=EXPORT_DEVICE)
test_candidate_idx = candidate_indices_for_hour(test_dataset, HOUR_OF_DAY)
print(f"2023 test pool: {len(test_candidate_idx)} starts | HOUR_OF_DAY={HOUR_OF_DAY}")
test_n = len(test_candidate_idx) if TEST_EXPORT_ALL else min(N_SAMPLES_TO_GEN, len(test_candidate_idx))
print(f"Will export: {test_n} test sequences")

for component in EXPORT_COMPONENTS:
    test_pfx, test_ch_names, test_tgt_idx = _component_export_settings(
        component, base=TEST_DATA_PREFIX_BASE
    )
    print(f"\n=== 2023 test export: {component} -> {test_pfx} ===")
    _run_one_export(
        test_pfx,
        test_ch_names,
        test_tgt_idx,
        start_time=TEST_START_TIME,
        end_time=TEST_END_TIME,
        n_samples_to_gen=test_n,
        crops_per_anchor=1,
        fixed_dataset_indices=tuple(int(x) for x in test_candidate_idx) if TEST_EXPORT_ALL else None,
    )


2023 test pool: 319 starts | HOUR_OF_DAY=12
Will export: 319 test sequences

=== 2023 test export: uvsshbathy -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_23_uvsshbathy_dt360 ===
Wrote config: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_23_uvsshbathy_dt360_export_config.json
Started background export PID 1973060 -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128_uvsshbathy/sample_23_uvsshbathy_dt360
Fresh export (log cleared, partial state reset if config changed)
Log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_uvsshbathy_sample_23_uvsshbathy_dt360.log
Monitor: tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_uvsshbathy_sample_23_uvsshbathy_dt360.log
Resume: python scripts/export_random_window_data.py --output-prefix /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_1